# Quick Checks on mounted Jupyter Kernel

Example case is based on Jupyter Server served on Kaggle (specifically, AIMO Comp). Notebook includes

- Monitor GPU/CPU usage
- Monitor PyTorch (and dependent libraries like Hugging Face Transformers, Sentence Transformers etc.) Caches and States
- Monitoring installed libraries in environment. This is particularly useful when continuing existing works in spin up containers that does not persist states and avoids re-installing libraries

Case Usage:

- Mounted Kernels from external providers e.g. Kaggle Hosted Jupyter Servers
- Manually monitoring PyTorch Models and running devices

# Workspace Configuration 

In [1]:
!nvidia-smi
#!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

Mon Mar 30 09:46:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Reset GPU Caches (PyTorch)

In [2]:
# PyTorch GPU Monitor Basics

import torch 

def gpu_info():
    assert torch.cuda.is_available(), "No GPU detected"
    
    print(f"GPUs available: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name} | {props.total_memory / 1e9:.2f} GB")
    
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM free: {free / 1e9:.2f} GB / {total / 1e9:.2f} GB")
    print(torch.cuda.memory_summary())

def reset_cuda():
    import gc, torch
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    
# gpu_info()

# Environment Installed Libraries

In [2]:
# Installs Packages where not already installed 

import subprocess
import sys

import importlib.util

required_packages = ["transformers", "accelerate", "torch"]
missing_packages = [p for p in required_packages if importlib.util.find_spec(p) is None]

if missing_packages:
    print(f"Installing missing packages: {missing_packages}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
else:
    print("All required packages are already installed.")

All required packages are already installed.
